ESM_C_Embeddings

In [ ]:
import os, pickle, torch
from tqdm.notebook import tqdm
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

FASTA_FILE = '/home/houxuc/snap/Topt/data/output_cdhit70.fasta'
OUTPUT_DIR = '../esmc_600m_features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
client = ESMC.from_pretrained('esmc_600m').to(device).eval()

# Parse FASTA (handles multi-line sequences)
sequences = {}
current_id, current_seq = None, []
with open(FASTA_FILE) as f:
    for line in f:
        line = line.strip()
        if line.startswith('>'):
            if current_id:
                sequences[current_id] = ''.join(current_seq)
            parts = line[1:].split('|')
            current_id = parts[1] if len(parts) > 1 else parts[0]
            current_seq = []
        else:
            current_seq.append(line)
    if current_id:
        sequences[current_id] = ''.join(current_seq)

print(f'Found {len(sequences)} sequences')

# Embed
for uid, seq in tqdm(sequences.items()):
    out_path = os.path.join(OUTPUT_DIR, f'{uid}.pkl')
    if os.path.exists(out_path):
        continue
    try:
        with torch.inference_mode():
            protein = ESMProtein(sequence=seq)
            protein_tensor = client.encode(protein)
            output = client.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
            emb = output.embeddings[0][1:-1].cpu()  # (L, D) — strip BOS/EOS
        with open(out_path, 'wb') as f:
            pickle.dump(emb, f)
    except Exception as e:
        print(f'ERROR [{uid}]: {e}')